<a href="https://colab.research.google.com/github/lee-doris/CPSAB/blob/main/20260816.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import numpy as np

# 1. 讀取 Excel（指定 header=3 讓年份對齊欄位）
url = 'https://github.com/lee-doris/CPSAB/raw/refs/heads/main/data/CPSAB_2023.xlsx' # 或檔案路徑
df = pd.read_excel(url, sheet_name='MDCR SUMMARY AB 1_CPS_11SAB', header=3)

# 2. 清理欄位與字串格式
df.columns = df.columns.astype(str).str.strip()
first_col = df.columns[0]

# 將年份欄位找出來
year_cols = [col for col in df.columns if col.isdigit()]

# 3. 將年份欄位強制轉為數字，無法轉換的（如空白字串）會變成 NaN
for col in year_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. 精準判定標頭列 (Header Row)：
# 條件：第一欄有文字，且所有年份欄位都是 NaN
is_header = df[first_col].notna() & (df[first_col].astype(str).str.strip() != 'BLANK') & df[year_cols].isna().all(axis=1)

# 5. 建立 Metric_Type 欄位並向下填補 (ffill)
df['Metric_Type'] = np.where(is_header, df[first_col], np.nan)
df['Metric_Type'] = df['Metric_Type'].ffill()

# 6. 過濾掉標頭列本身以及 BLANK 列
df_clean = df[~is_header].copy()
df_clean = df_clean[df_clean[first_col].astype(str).str.strip() != 'BLANK'].dropna(subset=[first_col]).reset_index(drop=True)

# 7. 轉成長格式 (pd.melt)
df_final = pd.melt(
    df_clean,
    id_vars=['Metric_Type', first_col],  # 務必包含 Metric_Type
    value_vars=year_cols,
    var_name='Year',
    value_name='Value'
).dropna(subset=['Value']).reset_index(drop=True)

# 顯示前 10 筆檢查
print(df_final.head(10))

                             Metric_Type  \
0  Number of Original Medicare Enrollees   
1  Number of Original Medicare Enrollees   
2  Number of Original Medicare Enrollees   
3               Persons With Utilization   
4               Persons With Utilization   
5               Persons With Utilization   
6               Persons With Utilization   
7               Persons With Utilization   
8               Persons With Utilization   
9               Persons With Utilization   

            Type of Coverage and Service  Year       Value  
0                   Part A and/or Part B  2018  38665082.0  
1                                 Part A  2018  38351612.0  
2                                 Part B  2018  33366109.0  
3                   Part A and/or Part B  2018  34822101.0  
4                                Part A   2018   7626520.0  
5            Inpatient Hospital Services  2018   6459625.0  
6      Skilled Nursing Facility Services  2018   1704557.0  
7                       Hos

In [29]:
import pandas as pd
import numpy as np

# 1. 整理單張 Sheet 的主函數
def clean_cms_sheet(url_or_path, sheet_name, group_name):
    df = pd.read_excel(url_or_path, sheet_name=sheet_name, header=3)

    # 清理欄位格式
    df.columns = df.columns.astype(str).str.strip()
    first_col = df.columns[0]
    year_cols = [col for col in df.columns if col.isdigit()]

    for col in year_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 判斷大類別標頭 (Metric_Type)
    is_header = (
        df[first_col].notna() &
        (df[first_col].astype(str).str.strip() != 'BLANK') &
        df[year_cols].isna().all(axis=1)
    )

    df['Metric_Type'] = np.where(is_header, df[first_col], np.nan)
    df['Metric_Type'] = df['Metric_Type'].ffill()

    # 剔除標頭與空行
    df_clean = df[~is_header].copy()
    df_clean = df_clean[df_clean[first_col].astype(str).str.strip() != 'BLANK'].dropna(subset=[first_col]).reset_index(drop=True)

    # 轉成長格式 (pd.melt)
    df_final = pd.melt(
        df_clean,
        id_vars=['Metric_Type', first_col],
        value_vars=year_cols,
        var_name='Year',
        value_name='Value'
    ).dropna(subset=['Value']).reset_index(drop=True)

    # 標記族群標籤
    df_final['Beneficiary_Group'] = group_name

    return df_final

# 2. 定義要處理的三個分頁與對應標籤
file_path = 'CPSAB_2023.xlsx' # 或傳入你的 URL

sheets_info = [
    ('MDCR SUMMARY AB 1_CPS_11SAB', 'All'),
    ('MDCR SUMMARY AB 2_CPS_11SAB', 'Aged'),
    ('MDCR SUMMARY AB 3_CPS_11SAB', 'Disabled')
]

# 3. 批次處理並進行垂直合併 (pd.concat)
df_list = [clean_cms_sheet(file_path, sheet, group) for sheet, group in sheets_info]
df_combined = pd.concat(df_list, ignore_index=True)

# 觀察最終合併結果
print(df_combined.head(10))
print(df_combined.tail(10))

                             Metric_Type  \
0  Number of Original Medicare Enrollees   
1  Number of Original Medicare Enrollees   
2  Number of Original Medicare Enrollees   
3               Persons With Utilization   
4               Persons With Utilization   
5               Persons With Utilization   
6               Persons With Utilization   
7               Persons With Utilization   
8               Persons With Utilization   
9               Persons With Utilization   

            Type of Coverage and Service  Year       Value Beneficiary_Group  
0                   Part A and/or Part B  2018  38665082.0               All  
1                                 Part A  2018  38351612.0               All  
2                                 Part B  2018  33366109.0               All  
3                   Part A and/or Part B  2018  34822101.0               All  
4                                Part A   2018   7626520.0               All  
5            Inpatient Hospital Services 

In [30]:
# 存成 CSV 檔 (index=False 代表不保留最左邊的數字索引，utf_8_sig 防止中文亂碼)
df_combined.to_csv('Medicare_2018_2023_Cleaned.csv', index=False, encoding='utf_8_sig')

In [31]:
from google.colab import files

files.download('Medicare_2018_2023_Cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>